# Piper nigrum — Reproducible K-means clustering

49 original molecular records, three descriptors, K-means evaluation, PCA visualization and seed stability. Requires original `ml.xlsx`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from google.colab import files

out = Path('outputs')
out.mkdir(exist_ok=True)
if not Path('ml.xlsx').exists():
    print('Upload original ml.xlsx')
    files.upload()
assert Path('ml.xlsx').exists(), 'Required input: ml.xlsx'

features = ['Gap_eV', 'BDE_MIN_kcal_moL', 'Carga_no_C']
df = pd.read_excel('ml.xlsx', sheet_name='Analise_Oleo_Pimenta', nrows=49)
assert len(df) == 49, f'Expected 49 rows, found {len(df)}'
assert all(c in df.columns for c in features), 'Missing required feature column'
for c in features:
    df[c] = pd.to_numeric(df[c].astype(str).str.strip().str.replace(',', '.', regex=False), errors='coerce')
missing = df[features].isna().sum()
print('Missing values by descriptor:')
print(missing)
assert missing.sum() == 0, 'Review missing descriptors; do not silently remove samples'
assert df[features].replace([np.inf,-np.inf],np.nan).notna().all().all(), 'Infinite values present'
assert df[features].nunique().ge(2).all(), 'Constant feature present'
print('Records:',len(df))

X = StandardScaler().fit_transform(df[features])
scan=[]
for k in range(2,9):
    m=KMeans(n_clusters=k,random_state=42,n_init=20)
    labels=m.fit_predict(X)
    scan.append({'k':k,'silhouette':float(silhouette_score(X,labels)), 'inertia':float(m.inertia_)})
scan=pd.DataFrame(scan)
print('K scan:')
display(scan)
scan.to_csv(out/'k_selection.csv',index=False)
selected_k = int(scan.loc[scan.silhouette.idxmax(),'k'])
assert selected_k==2, f'Expected K=2; found K={selected_k}. Check source data and preprocessing.'
model=KMeans(n_clusters=2,random_state=42,n_init=20)
labels=model.fit_predict(X)
score=float(silhouette_score(X,labels))
pca=PCA(n_components=2)
coords=pca.fit_transform(X)
result=df.copy()
result['Cluster']=labels
result['PC1']=coords[:,0]
result['PC2']=coords[:,1]
result.to_csv(out/'cluster_assignments.csv',index=False)
result.groupby('Cluster')[features].agg(['count','mean','std','min','max']).to_csv(out/'cluster_descriptives.csv')
print('Cluster sizes:')
print(result.Cluster.value_counts().sort_index())
print('Cluster means:')
display(result.groupby('Cluster')[features].mean())

fig,ax=plt.subplots(figsize=(7,5))
for label in sorted(set(labels)):
    mask=labels==label
    ax.scatter(coords[mask,0],coords[mask,1],s=55,label=f'Cluster {label}',alpha=0.85)
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',title='Piper nigrum — K-means, PCA projection')
ax.legend(); ax.grid(alpha=.2);fig.tight_layout();fig.savefig(out/'pca_clusters.png',dpi=180);plt.show()

runs=[KMeans(n_clusters=2,random_state=s,n_init=20).fit_predict(X) for s in range(30)]
def partition_key(a):
    return tuple(sorted(tuple(np.flatnonzero(a==label)) for label in np.unique(a)))
unique_partitions=len({partition_key(r) for r in runs})
aris=[float(adjusted_rand_score(runs[0],r)) for r in runs]
metrics={'n_compounds':len(df),'features':features,'best_k':2,'silhouette':score,'cluster_sizes':{str(k):int(v) for k,v in result.Cluster.value_counts().sort_index().items()},'pca_explained_variance':float(sum(pca.explained_variance_ratio_)),'stability_runs':len(runs),'unique_partitions':unique_partitions,'ari_min':min(aris),'ari_mean':float(np.mean(aris)),'ari_max':max(aris)}
(out/'clustering_metrics.json').write_text(json.dumps(metrics,indent=2))
print(json.dumps(metrics,indent=2))
print('Generated files:',*[p.name for p in out.iterdir()],sep='\n - ')
